# 00_setup.ipynb


## Objectif

Ce notebook sert à :

- vérifier l'environnement Python;
- tester les imports essentiels du projet;
- rendre le dossier `src/` importable;
- charger la configuration centrale;
- vérifier la structure du projet;
- créer les dossiers utiles ou nécessaire;
- vérifier la présence du dataset;
- charger les premières constantes du projet;

## Résultat attendu

À la fin de ce notebook, le projet doit être prêt pour :

- `01_cadre_metier_&_eda_audit.ipynb`
- `02_preprocessing.ipynb`
- `03_modeling.ipynb`
- `04_interpretation_business.ipynb`
- `05_reporting.ipynb`

In [4]:
# ==============================
# Imports de base
# ==============================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

#
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Réglages d'affichage
pd.set_option("display.max_columns", 21)
#pd.set_option("display.max_rows", 200)

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_style("whitegrid")

# Toujours afficher la dimension du dataframe
pd.set_option("display.show_dimensions", True)

print("Imports chargés avec succès.")
print("Python       :", sys.version.split("\n")[0])
print("NumPy        :", np.__version__)
print("Pandas       :", pd.__version__)
print("Scikit-learn :", sklearn.__version__)

Imports chargés avec succès.
Python       : 3.11.13 | packaged by conda-forge | (main, Jun  4 2025, 14:48:01) [Clang 18.1.8 ]
NumPy        : 1.26.4
Pandas       : 2.3.2
Scikit-learn : 1.3.2


In [5]:
# ==============================
# Chemins du projet
# ==============================

# Le notebook est exécuté depuis le dossier notebooks/
# On remonte donc d'un niveau pour retrouver la racine du projet.

project_root = Path.cwd().resolve().parents[0]
src_path = project_root / "src"

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("Racine du projet :", project_root)
print("Dossier src      :", src_path)

Racine du projet : /Users/dialloamadou/Desktop/projets_data_science/churn_prediction
Dossier src      : /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/src


In [6]:
# ==============================
# Import de la configuration et des fonctions utilitaires
# ==============================

from src.config import (
    PROJECT_ROOT,
    DATA_DIR,
    RAW_DATA_DIR,
    INTERIM_DATA_DIR,
    PROCESSED_DATA_DIR,
    NOTEBOOKS_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    MODELS_DIR,
    SRC_DIR,
    DATASET_FILENAME,
    RAW_DATA_PATH,
    TARGET_COL,
    ID_COL,
    RANDOM_STATE,
    TEST_SIZE,
    CV_FOLDS,
    N_JOBS,
    EXPECTED_COLUMNS,
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
)

from src.utils import ensure_directories, load_csv, describe_dataframe

# Alias locaux en minuscules pour rendre le notebook plus lisible
data_dir = DATA_DIR
raw_data_dir = RAW_DATA_DIR
interim_data_dir = INTERIM_DATA_DIR
processed_data_dir = PROCESSED_DATA_DIR
notebooks_dir = NOTEBOOKS_DIR
reports_dir = REPORTS_DIR
figures_dir = FIGURES_DIR
tables_dir = TABLES_DIR
models_dir = MODELS_DIR

dataset_filename = DATASET_FILENAME
raw_data_path = RAW_DATA_PATH
target_col = TARGET_COL
id_col = ID_COL

random_state = RANDOM_STATE
test_size = TEST_SIZE
cv_folds = CV_FOLDS
n_jobs = N_JOBS

categorical_features = CATEGORICAL_FEATURES
numerical_features = NUMERICAL_FEATURES

print("Configuration et utilitaires chargés avec succès.")

Configuration et utilitaires chargés avec succès.


In [7]:
# ==============================
# Vérification / création des dossiers utiles
# ==============================

required_dirs = [
    data_dir,
    raw_data_dir,
    interim_data_dir,
    processed_data_dir,
    notebooks_dir,
    reports_dir,
    figures_dir,
    tables_dir,
    models_dir,
    SRC_DIR,
]

ensure_directories(required_dirs)

print("Dossiers vérifiés / créés :")
for directory in required_dirs:
    print("-", directory)

Dossiers vérifiés / créés :
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/data
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/data/raw
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/data/interim
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/data/processed
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/notebooks
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/reports
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/reports/figures
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/reports/tables
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/models
- /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/src


In [8]:
# ==============================
# Vérification du dataset et chargement
# ==============================

print("Nom du dataset attendu :", dataset_filename)
print("Chemin attendu         :", raw_data_path)

if raw_data_path.exists():
    print("\n✅ Dataset trouvé.")
else:
    raise FileNotFoundError(
        f"❌ Dataset introuvable. Merci de placer le fichier dans : {raw_data_path}"
    )

df = load_csv(raw_data_path)

print("\nDataset chargé avec succès.")
print("Dimensions :", df.shape)

#display(df.head())

Nom du dataset attendu : WA_Fn-UseC_-Telco-Customer-Churn.csv
Chemin attendu         : /Users/dialloamadou/Desktop/projets_data_science/churn_prediction/data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv

✅ Dataset trouvé.

Dataset chargé avec succès.
Dimensions : (7043, 21)
